# Trade Execution

Runs execution strategies against the 5-minute `BINNED_DATA` bins and analyses the results (fill rate, implementation shortfall, cost).

All reusable logic lives in [`execution.py`](execution.py); this notebook is the driver. (`test_snowflake.ipynb` stays the data explorer.)

**Layers** (see `execution.py` docstrings for full detail):

| Function | What it does |
|---|---|
| `simulate_bin_fill` | scalar reference model for a single bin |
| `twap_schedule` | baseline strategy: split quantity evenly across the day's bins |
| `vwap_schedule` | baseline strategy: split quantity by each bin's share of the day's volume |
| `run_strategy` | run one order `(security, date, side, quantity)` → per-bin fills |
| `run_trade_list` | run many orders (the trade list) → per-bin fills with `order_id` |
| `summarise_fills` | collapse to one row per order with fill rate / IS / cost |


In [17]:
import polars as pl
from execution import (
    twap_schedule,
    vwap_schedule,
    run_strategy,
    run_trade_list,
    summarise_fills,
)

pl.Config.set_tbl_cols(-1)
RAW = "data/raw"

trade_list = pl.read_csv(f"{RAW}/trade_list.csv")
trade_list.head()

security,date,trade_list,side,quantity
str,str,str,str,i64
"""IK2026M Comdty""","""2026-03-06""","""small_buys""","""buy""",333
"""IK2026M Comdty""","""2026-03-09""","""small_buys""","""buy""",334
"""IK2026M Comdty""","""2026-03-10""","""small_buys""","""buy""",333
"""IK2026M Comdty""","""2026-03-11""","""small_buys""","""buy""",339
"""IK2026M Comdty""","""2026-03-12""","""small_buys""","""buy""",343


## Load bins

For a quick demo we load bins for just a couple of instruments (one liquid, one illiquid). For a full back-test, read the whole table instead — it is ~13M rows / ~2.5 GB in memory:

```python
bins = pl.read_csv(f"{RAW}/binned_data.csv")
```

In [18]:
demo_secs = ["GC2025V Comdty", "PA2022Z Comdty"]  # liquid gold, illiquid palladium

bins = (
    pl.scan_csv(f"{RAW}/binned_data.csv")
    .filter(pl.col("security").is_in(demo_secs))
    .collect()
)
bins

qcode,publication_date,security,bin_start_time,gmt_offset_hours,open,high,low,close,twa_bid_size,twa_ask_size,bid_size_start,bid_size_end,ask_size_start,ask_size_end,volume,signed_volume,trade_count,vwap,twa_bid,twa_ask,bid_start,bid_end,ask_start,ask_end
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""GC""","""2025-08-21""","""GC2025V Comdty""","""12:30:00.000000000""",-4.0,3357.0,3357.2,3356.9,3357.1,3.72327,4.416947,7,2,4,1,5,3,5,3357.06,3357.106662,3357.428751,3357.4,3357.0,3357.8,3357.2
"""GC""","""2025-08-21""","""GC2025V Comdty""","""13:05:00.000000000""",-4.0,3358.0,3358.3,3358.0,3358.3,3.769527,4.040703,2,4,5,4,2,-1,2,3358.15,3358.166469,3358.434343,3357.5,3357.9,3357.7,3358.2
"""GC""","""2025-09-26""","""GC2025V Comdty""","""07:15:00.000000000""",-4.0,3743.7,3745.3,3743.3,3745.3,3.585983,3.639973,3,2,5,6,12,7,12,3743.933333,3744.485734,3744.780858,3743.4,3746.2,3743.7,3746.5
"""GC""","""2025-09-05""","""GC2025V Comdty""","""07:30:00.000000000""",-4.0,3576.6,3576.6,3575.6,3575.8,3.80029,3.624047,8,2,2,1,15,2,11,3576.033333,3575.999761,3576.24569,3576.3,3575.8,3576.6,3576.0
"""GC""","""2025-09-05""","""GC2025V Comdty""","""14:10:00.000000000""",-4.0,3621.0,3621.0,3619.2,3619.2,3.153997,3.67401,2,7,2,6,7,2,7,3620.071429,3621.03912,3621.265013,3622.2,3619.0,3622.4,3619.3
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""PA""","""2022-10-20""","""PA2022Z Comdty""","""04:10:00.000000000""",-4.0,1998.0,2001.0,1987.5,1995.0,1.256073,1.59502,1,1,1,2,17,2,15,1996.941176,1993.873782,1998.529693,1991.0,1992.5,1996.0,1997.0
"""PA""","""2022-10-20""","""PA2022Z Comdty""","""04:30:00.000000000""",-4.0,1992.0,1992.0,1992.0,1992.0,1.20146,2.1444,1,1,1,9,1,0,1,1992.0,1989.456387,1993.66039,1987.5,1991.5,1992.0,1996.0
"""PA""","""2022-09-28""","""PA2022Z Comdty""","""15:40:00.000000000""",-4.0,2156.0,2158.0,2155.5,2156.5,1.126773,1.634757,1,1,1,1,6,2,4,2156.25,2154.504405,2157.68185,2154.5,2154.0,2157.5,2158.5


## Evaluation metrics

Every strategy below is scored the same way: `summarise_fills` collapses the per-bin fills to **one row per order**. The three headline numbers (full column-by-column reference in [`docs/execution_metrics.md`](docs/execution_metrics.md)):

- **`fill_rate`** = `total_filled / order_quantity` — fraction of the order that actually got done. A strategy that allocates to zero-volume bins can't fill there, so illiquid names (e.g. `PA`) under-fill — the gap a smarter schedule should close.
- **`exec_slippage_bps`** — filled-only slippage vs the arrival price, signed so **positive = worse**.
- **`is_bps`** — implementation shortfall vs arrival price, including opportunity cost on the unfilled remainder marked at the day's terminal price. Captures both spread *and* intraday price drift, and — unlike the other two — it penalises a low `fill_rate`, so it's the right top-line score when comparing strategies.

**Plugging in a new strategy:** write a function `my_strategy(bins, quantity, **params) -> Sequence[float]` returning the per-bin requested quantities, and pass `strategy=my_strategy` to `run_strategy` / `run_trade_list`. Everything downstream (fill model, metrics) stays the same — only the per-bin allocation changes, which is what makes the strategies comparable.

## TWAP

**TWAP (Time-Weighted Average Price)** splits the order **evenly across every bin** in the day — `quantity / n_bins` each — regardless of how much volume a bin actually had. It's the naive baseline: it allocates to zero-volume / no-trade bins too, so that quantity never fills and is lost. The under-filling this causes on illiquid or short days is exactly what a smarter schedule should beat.

### Single order

Take one real order (one bucket for a security-day) and run it through TWAP. `run_strategy` takes the bins for that `(security, date)` plus the order's `side` and `quantity`, and returns one row per bin.

In [19]:
# pick one real order from the trade list (one row = one bucket for a security-day)
order = trade_list.filter(pl.col("security") == "GC2025V Comdty").row(0, named=True)
print(order)

day = bins.filter(
    (pl.col("security") == order["security"])
    & (pl.col("publication_date") == order["date"])
)

fills = run_strategy(day, side=order["side"], quantity=order["quantity"], strategy=twap_schedule)
fills.head()

{'security': 'GC2025V Comdty', 'date': '2025-07-31', 'trade_list': 'large_buys', 'side': 'buy', 'quantity': 761}


order_id,security,publication_date,qcode,side,order_quantity,bin_start_time,volume,open,vwap,twa_bid,twa_ask,spread,q_requested,q_filled,q_unfilled,participation_rate,slippage_factor,fill_price,notional,cost,filled,data_available
i32,str,str,str,str,f64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:00:00.000000000""",23,3323.0,3323.421739,3322.799493,3323.088507,0.289015,4.847134,4.847134,0.0,0.210745,0.229825,3323.488162,16109.391664,0.32196,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:05:00.000000000""",233,3323.1,3324.727468,3323.683917,3323.962368,0.27845,4.847134,4.847134,0.0,0.020803,0.140789,3324.766671,16115.588766,0.190021,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:10:00.000000000""",165,3323.7,3324.784848,3324.371042,3324.66078,0.289738,4.847134,4.847134,0.0,0.029377,0.148471,3324.827866,16115.885389,0.208512,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:15:00.000000000""",19,3323.6,3323.752632,3323.940178,3324.289575,0.349397,4.847134,4.847134,0.0,0.255112,0.242838,3323.837479,16111.084848,0.411264,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:20:00.000000000""",36,3325.4,3325.483333,3325.593179,3325.900623,0.307444,4.847134,4.847134,0.0,0.134643,0.20377,3325.545981,16119.366189,0.303662,true,true


In [20]:
summarise_fills(fills)

order_id,security,date,qcode,side,order_quantity,n_bins,n_fillable,n_filled,total_volume,total_requested,total_filled,filled_notional,total_cost,arrival_price,terminal_price,fill_rate,avg_fill_price,unfilled_qty,participation_overall,exec_slippage_bps,is_currency,is_bps
i32,str,str,str,str,f64,u32,u32,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,157,156,156,7973,761.0,750.764331,2.4970e6,48.263654,3323.0,3316.984615,0.98655,3325.94194,10.235669,0.094163,8.853264,2147.131789,8.490704


### The whole trade list

The `TRADE_LIST` table has six rows per (security, date) — `small_buys`, `small_sells`, `medium_buys`, `medium_sells`, `large_buys`, `large_sells`. Pass `trade_list=` to run **one bucket at a time** (the usual back-test). Each order gets a unique `order_id` and is auto-tagged with its bucket.

Here TWAP runs the `medium_buys` bucket. Watch the illiquid `PA` rows under-fill (`fill_rate < 1`): TWAP keeps allocating to bins with no volume to fill against.

In [21]:
orders = trade_list.filter(pl.col("security").is_in(demo_secs))

# run a single bucket, e.g. medium_buys (one of the six)
all_fills = run_trade_list(bins, orders, trade_list="medium_buys", strategy=twap_schedule)
all_fills.shape

(16929, 24)

In [22]:
summary = summarise_fills(all_fills)
summary.sort("security", "side")

order_id,trade_list,security,date,qcode,side,order_quantity,n_bins,n_fillable,n_filled,total_volume,total_requested,total_filled,filled_notional,total_cost,arrival_price,terminal_price,fill_rate,avg_fill_price,unfilled_qty,participation_overall,exec_slippage_bps,is_currency,is_bps
i32,str,str,str,str,str,f64,u32,u32,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
65,"""medium_buys""","""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",76.0,157,156,156,7973,76.0,75.515924,251156.085013,3.011831,3323.0,3316.984615,0.993631,3325.869209,0.484076,0.009471,8.634395,213.759094,8.464098
107,"""medium_buys""","""GC2025V Comdty""","""2025-09-29""","""GC""","""buy""",50.0,157,124,124,965,50.0,39.490446,150714.511803,2.322219,3812.3,3821.3,0.789809,3816.48038,10.509554,0.040923,10.965505,259.671038,13.622802
106,"""medium_buys""","""GC2025V Comdty""","""2025-09-26""","""GC""","""buy""",52.0,157,145,145,2439,52.0,48.025478,180544.953201,2.393536,3739.5,3763.743478,0.923567,3759.357779,3.974522,0.019691,53.102765,1050.035561,53.999175
105,"""medium_buys""","""GC2025V Comdty""","""2025-09-25""","""GC""","""buy""",54.0,157,156,156,4274,54.0,53.656051,200831.194049,2.317208,3738.3,3746.742857,0.993631,3742.936546,0.343949,0.012554,12.40282,251.682675,12.467673
104,"""medium_buys""","""GC2025V Comdty""","""2025-09-24""","""GC""","""buy""",55.0,157,156,156,4140,55.0,54.649682,205235.635349,2.314576,3773.6,3730.85,0.993631,3755.477244,0.350318,0.0132,-48.025112,-1005.378983,-48.440794
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6,"""medium_buys""","""PA2022Z Comdty""","""2022-09-08""","""PA""","""buy""",16.0,157,127,127,1941,16.0,12.942675,27191.483891,10.053829,2030.0,2138.277778,0.808917,2100.916816,3.057325,0.006668,349.343923,1248.893657,384.511594
5,"""medium_buys""","""PA2022Z Comdty""","""2022-09-07""","""PA""","""buy""",16.0,157,119,119,1287,16.0,12.127389,24345.456478,8.838248,1988.0,2033.5,0.757962,2007.477241,3.872611,0.009423,97.97405,412.411892,129.656656
4,"""medium_buys""","""PA2022Z Comdty""","""2022-09-06""","""PA""","""buy""",16.0,157,119,119,1435,16.0,12.127389,24260.940763,8.786417,2063.5,1984.75,0.757962,2000.508246,3.872611,0.008451,-305.266559,-1068.893632,-323.750192


## VWAP

**VWAP (Volume-Weighted Average Price)** splits the order across bins **in proportion to each bin's share of the day's traded volume** (`quantity × volume / total_volume`) instead of evenly. It tracks the intraday volume profile (heavy at the open/close, thin midday) and — crucially — allocates **nothing** to zero-volume bins, so it should fill better than TWAP on the illiquid names where naive TWAP wastes quantity on dead bins. Everything downstream (fill model, metrics) is identical; only the per-bin allocation changes.

### Single order

The same order as the TWAP single-order example above, now run through `vwap_schedule` — directly comparable.

In [23]:
# single order, VWAP — same order as the TWAP single-order example above
vwap_fills = run_strategy(day, side=order["side"], quantity=order["quantity"], strategy=vwap_schedule)
summarise_fills(vwap_fills)

order_id,security,date,qcode,side,order_quantity,n_bins,n_fillable,n_filled,total_volume,total_requested,total_filled,filled_notional,total_cost,arrival_price,terminal_price,fill_rate,avg_fill_price,unfilled_qty,participation_overall,exec_slippage_bps,is_currency,is_bps
i32,str,str,str,str,f64,u32,u32,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,157,156,156,7973,761.0,761.0,2.5320e6,39.801507,3323.0,3316.984615,1.0,3327.165484,0.0,0.095447,12.53531,3169.933001,12.53531


### The whole trade list

The same `medium_buys` bucket as the TWAP run above, now under VWAP. Compare the `fill_rate` on the illiquid `PA` rows: by allocating **nothing** to dead bins, VWAP recovers the quantity TWAP wasted there.

VWAP fills 100% here, but that isn't guaranteed — it only spreads the order over bins that *did* trade, so it still leaves quantity unfilled when the order is larger than the day can absorb (`quantity > 2 × total daily volume`, since each bin caps at `2 × volume`), on zero-volume days, or in the rare bins that have volume but missing price data. These demo orders are sized well within each name's daily liquidity, so none of those bite.

In [24]:
# run the same bucket through VWAP and summarise
vwap_all_fills = run_trade_list(bins, orders, trade_list="medium_buys", strategy=vwap_schedule)
vwap_summary = summarise_fills(vwap_all_fills)
vwap_summary.sort("security", "side")

order_id,trade_list,security,date,qcode,side,order_quantity,n_bins,n_fillable,n_filled,total_volume,total_requested,total_filled,filled_notional,total_cost,arrival_price,terminal_price,fill_rate,avg_fill_price,unfilled_qty,participation_overall,exec_slippage_bps,is_currency,is_bps
i32,str,str,str,str,str,f64,u32,u32,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
65,"""medium_buys""","""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",76.0,157,156,156,7973,76.0,76.0,252863.309002,2.707171,3323.0,3316.984615,1.0,3327.148803,-1.4211e-14,0.009532,12.485112,315.309002,12.485112
107,"""medium_buys""","""GC2025V Comdty""","""2025-09-29""","""GC""","""buy""",50.0,157,124,124,965,50.0,50.0,190879.862579,2.411802,3812.3,3821.3,1.0,3817.597252,0.0,0.051813,13.895159,264.862579,13.895159
106,"""medium_buys""","""GC2025V Comdty""","""2025-09-26""","""GC""","""buy""",52.0,157,145,145,2439,52.0,52.0,195558.433043,2.175888,3739.5,3763.743478,1.0,3760.739097,0.0,0.02132,56.796622,1104.433043,56.796622
105,"""medium_buys""","""GC2025V Comdty""","""2025-09-25""","""GC""","""buy""",54.0,157,156,156,4274,54.0,54.0,201976.895578,2.148549,3738.3,3746.742857,1.0,3740.312881,-7.1054e-15,0.012635,5.384482,108.695578,5.384482
104,"""medium_buys""","""GC2025V Comdty""","""2025-09-24""","""GC""","""buy""",55.0,157,156,156,4140,55.0,55.0,206349.700302,2.046075,3773.6,3730.85,1.0,3751.812733,-7.1054e-15,0.013285,-57.736027,-1198.299698,-57.736027
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6,"""medium_buys""","""PA2022Z Comdty""","""2022-09-08""","""PA""","""buy""",16.0,157,127,127,1941,16.0,16.0,33901.525938,10.072048,2030.0,2138.277778,1.0,2118.845371,0.0,0.008243,437.661927,1421.525938,437.661927
5,"""medium_buys""","""PA2022Z Comdty""","""2022-09-07""","""PA""","""buy""",16.0,157,119,119,1287,16.0,16.0,32110.730707,10.224103,1988.0,2033.5,1.0,2006.920669,0.0,0.012432,95.174392,302.730707,95.174392
4,"""medium_buys""","""PA2022Z Comdty""","""2022-09-06""","""PA""","""buy""",16.0,157,119,119,1435,16.0,16.0,31895.09215,10.046854,2063.5,1984.75,1.0,1993.443259,0.0,0.01115,-339.504437,-1120.90785,-339.504437
